# 02 — Modélisation, Évaluation & Interprétation
## Prédiction du Risque de Distribution des Médicaments en Tunisie

**3 algorithmes :** Régression Logistique · Random Forest · XGBoost  
**Optimisation :** RandomizedSearchCV  
**Métriques :** F1-Score Macro + AUC-ROC  
**Interprétation :** SHAP Values

---

## 1. Imports

## 2. Chargement des données

> Les données sont chargées depuis les fichiers sauvegardés
> par `01_EDA_Pretraitement.ipynb`.

> ⚠️ Exécuter `01_EDA_Pretraitement.ipynb` d'abord si les fichiers n'existent pas.

In [1]:
from sklearn.linear_model    import LogisticRegression
from sklearn.ensemble        import RandomForestClassifier
from xgboost                 import XGBClassifier
from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold
from sklearn.metrics         import (
    classification_report, confusion_matrix,
    f1_score, roc_auc_score, roc_curve,
    ConfusionMatrixDisplay
)

In [ ]:
import pickle

print("📥 Chargement des données depuis Google Drive...")

with open('/content/drive/MyDrive/data_ml/X_train_sm.pkl', 'rb') as f:
      X_train_sm = pickle.load(f)

with open('/content/drive/MyDrive/data_ml/y_train_sm.pkl', 'rb') as f:
      y_train_sm = pickle.load(f)

with open('/content/drive/MyDrive/data_ml/X_test.pkl', 'rb') as f:
      X_test = pickle.load(f)

with open('/content/drive/MyDrive/data_ml/y_test.pkl', 'rb') as f:
      y_test = pickle.load(f)

print(f" Données chargées avec succès!")
print(f"   X_train_sm: {X_train_sm.shape}")
print(f"   y_train_sm: {y_train_sm.shape}")
print(f"   X_test: {X_test.shape}")
print(f"   y_test: {y_test.shape}")

✅ Données chargées avec succès !
   X_train_sm : (7010, 28)
   y_train_sm : (7010,)
   X_test     : (1211, 28)
   y_test     : (1211,)


In [ ]:
for var in ['X_train_sm','y_train_sm','X_test','y_test']:
    try:
        v = eval(var)
        print(f'{var} : {v.shape}')
    except NameError:
        print(f' {var} manquant — exécuter 01_EDA_Pretraitement.ipynb')

X_train_sm : (7010, 28)
y_train_sm : (7010,)
X_test : (1211, 28)
y_test : (1211,)


## 3. Modélisation

### 3.1 Justification des 3 algorithmes

| Algorithme | Type | Justification |
|-----------|------|---------------|
| Régression Logistique | **Linéaire** | Baseline simple et interprétable — référence |
| Random Forest | **Ensembliste (bagging)** | Robuste, gère la non-linéarité et les outliers |
| XGBoost | **Ensembliste (boosting)** | Meilleure performance sur données tabulaires |

### 3.2 Séparation Train/Test

> Le split a été effectué dans le notebook 01 (section 5.3) :
> **80% train** (~4842 médicaments) · **20% test** (~1211 médicaments)
> avec `stratify=y` pour maintenir le ratio 72/28.

### 3.3 Modèle 1 — Régression Logistique (baseline)

In [4]:
SEED=42


lr = LogisticRegression(
    class_weight='balanced',
    max_iter=1000,
    random_state=SEED
)
lr.fit(X_train_sm, y_train_sm)

y_pred_lr  = lr.predict(X_test)
y_proba_lr = lr.predict_proba(X_test)[:, 1]

print('=== RÉGRESSION LOGISTIQUE ===')
print(classification_report(y_test, y_pred_lr,
      target_names=['Non distribué','Distribué']))
print(f'F1-Score Macro : {f1_score(y_test, y_pred_lr, average="macro"):.4f}')
print(f'AUC-ROC        : {roc_auc_score(y_test, y_proba_lr):.4f}')

=== RÉGRESSION LOGISTIQUE ===
               precision    recall  f1-score   support

Non distribué       0.42      0.57      0.48       335
    Distribué       0.81      0.70      0.75       876

     accuracy                           0.67      1211
    macro avg       0.62      0.64      0.62      1211
 weighted avg       0.70      0.67      0.68      1211

F1-Score Macro : 0.6190
AUC-ROC        : 0.6836


### 3.4 Modèle 2 — Random Forest + RandomizedSearchCV

In [ ]:
param_rf = {
    'n_estimators'      : [100, 200, 300],
    'max_depth'         : [None, 10, 20, 30],
    'min_samples_split' : [2, 5, 10],
    'min_samples_leaf'  : [1, 2, 4],
    'class_weight'      : ['balanced']
}
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)

rf_search = RandomizedSearchCV(
    RandomForestClassifier(random_state=SEED),
    param_distributions=param_rf,
    n_iter=20, scoring='f1_macro',
    cv=cv, random_state=SEED, n_jobs=-1, verbose=1
)
rf_search.fit(X_train_sm, y_train_sm)
print(f'Meilleurs paramètres : {rf_search.best_params_}')

rf = rf_search.best_estimator_
y_pred_rf  = rf.predict(X_test)
y_proba_rf = rf.predict_proba(X_test)[:, 1]

print('\n=== RANDOM FOREST ===')
print(classification_report(y_test, y_pred_rf,
      target_names=['Non distribué','Distribué']))
print(f'F1-Score Macro : {f1_score(y_test, y_pred_rf, average="macro"):.4f}')
print(f'AUC-ROC        : {roc_auc_score(y_test, y_proba_rf):.4f}')

### 3.5 Modèle 3 — XGBoost + RandomizedSearchCV

In [ ]:
param_xgb = {
    'n_estimators'    : [100, 200, 300],
    'max_depth'       : [3, 5, 7, 9],
    'learning_rate'   : [0.01, 0.05, 0.1, 0.2],
    'subsample'       : [0.7, 0.8, 1.0],
    'colsample_bytree': [0.7, 0.8, 1.0]
}
scale_pos = (y_train_sm==0).sum() / (y_train_sm==1).sum()

xgb_search = RandomizedSearchCV(
    XGBClassifier(scale_pos_weight=scale_pos, random_state=SEED,
                  eval_metric='logloss', verbosity=0),
    param_distributions=param_xgb,
    n_iter=20, scoring='f1_macro',
    cv=cv, random_state=SEED, n_jobs=-1, verbose=1
)
xgb_search.fit(X_train_sm, y_train_sm)
print(f'Meilleurs paramètres : {xgb_search.best_params_}')

xgb = xgb_search.best_estimator_
y_pred_xgb  = xgb.predict(X_test)
y_proba_xgb = xgb.predict_proba(X_test)[:, 1]

print('\n=== XGBOOST ===')
print(classification_report(y_test, y_pred_xgb,
      target_names=['Non distribué','Distribué']))
print(f'F1-Score Macro : {f1_score(y_test, y_pred_xgb, average="macro"):.4f}')
print(f'AUC-ROC        : {roc_auc_score(y_test, y_proba_xgb):.4f}')

## 4. Évaluation et Optimisation

### 4.1 Choix des métriques

| Métrique | Pourquoi |
|----------|----------|
| **F1-Score Macro** | Déséquilibre 72/28 — l'accuracy seule est trompeuse |
| **AUC-ROC** | Mesure la capacité de discrimination entre les deux classes |

In [ ]:
results = pd.DataFrame({
    'Modèle'        : ['Régression Logistique','Random Forest','XGBoost'],
    'F1-Score Macro': [
        f1_score(y_test, y_pred_lr,  average='macro'),
        f1_score(y_test, y_pred_rf,  average='macro'),
        f1_score(y_test, y_pred_xgb, average='macro')
    ],
    'AUC-ROC': [
        roc_auc_score(y_test, y_proba_lr),
        roc_auc_score(y_test, y_proba_rf),
        roc_auc_score(y_test, y_proba_xgb)
    ]
}).round(4).sort_values('F1-Score Macro', ascending=False)

print(results.to_string(index=False))

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
colors = ['#3498db','#2ecc71','#e74c3c']
for ax, metric in zip(axes, ['F1-Score Macro','AUC-ROC']):
    bars = ax.bar(results['Modèle'], results[metric], color=colors, edgecolor='white')
    for bar, val in zip(bars, results[metric]):
        ax.text(bar.get_x()+bar.get_width()/2,
                bar.get_height()+0.005, f'{val:.4f}',
                ha='center', fontweight='bold')
    ax.set_title(metric, fontweight='bold')
    ax.set_ylim(0, 1.1)
    ax.set_xticklabels(results['Modèle'], rotation=15, ha='right')
plt.suptitle('Comparaison des 3 modèles', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

### 4.2 Matrices de Confusion

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, (name, y_pred) in zip(axes, [
    ('Régression Logistique', y_pred_lr),
    ('Random Forest',         y_pred_rf),
    ('XGBoost',               y_pred_xgb)
]):
    cm = confusion_matrix(y_test, y_pred)
    ConfusionMatrixDisplay(cm,
        display_labels=['Non distribué','Distribué']).plot(
        ax=ax, colorbar=False, cmap='Blues')
    ax.set_title(name, fontweight='bold')
plt.suptitle('Matrices de Confusion', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## 5. Interprétation — SHAP Values

> SHAP (SHapley Additive exPlanations) explique **pourquoi** le modèle
> a prédit chaque valeur. Il quantifie la contribution de chaque feature.

> On utilise le meilleur modèle (XGBoost) pour l'interprétation.

In [ ]:
explainer   = shap.TreeExplainer(xgb)
shap_values = explainer.shap_values(X_test)

# Importance globale
plt.figure(figsize=(10, 8))
shap.summary_plot(shap_values, X_test,
                  plot_type='bar', max_display=15, show=False)
plt.title('SHAP — Importance globale des features', fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Explication individuelle (waterfall)
idx = 0
print(f'Médicament {idx} — Réel : {y_test.iloc[idx]} | Prédit : {y_pred_xgb[idx]}')
shap.waterfall_plot(
    shap.Explanation(
        values        = shap_values[idx],
        base_values   = explainer.expected_value,
        data          = X_test.iloc[idx],
        feature_names = X_test.columns.tolist()
    )
)

## 6. Conclusion — Performance et Applicabilité

In [ ]:
print('='*55)
print('              CONCLUSION FINALE')
print('='*55)
print(results.to_string(index=False))
print(f'\n Meilleur modèle  : XGBoost')
print(f' Optimisation     : RandomizedSearchCV (20 iter, CV=5 folds)')
print(f' Déséquilibre     : SMOTE + class_weight')
print(f' Métriques        : F1-Score Macro + AUC-ROC')
print(f' Interprétation   : SHAP Values')
print(f'\nApplicabilité réelle :')
print(f'  Un labo soumet les caractéristiques dun nouveau médicament')
print(f'  → le modèle prédit la probabilité darriver en pharmacie')
print(f'  → SHAP explique les raisons de la prédiction')
print(f'\nLimites connues :')
print(f'  — Cold Start Problem pour les nouveaux laboratoires')
print(f'  — Dataset limité à la Tunisie (non généralisable)')